# 06 — Full Pipeline Evaluation

**Goal:** Wire everything together, fit the RDS policy on training data, evaluate on test data using **importance sampling**, and measure **lift** over the random baseline.

This notebook answers:
- Does the full RDS pipeline actually beat random selection?
- How does importance sampling work for offline evaluation?
- What lift do we achieve, and how does it compare to Duolingo's reported ~1.9%?

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_sample
from src.bandit.rds_policy import RDSPolicy
from src.evaluation.baseline import compute_random_baseline, compute_lift
from src.evaluation.importance_sampling import (
    compute_importance_weights,
    weighted_importance_sampling,
)

sns.set_style("whitegrid")
print("Ready!")

---
## 1. Load Train & Test Samples

In [ ]:
train_df = load_sample(n_rows=100_000, split="train")
test_df = load_sample(n_rows=100_000, split="test")

print(f"\nTrain: {len(train_df):,} rows")
print(f"Test:  {len(test_df):,} rows")

---
## 2. Fit the RDS Policy

This runs the full training pipeline in one call:
1. Compute per-template reward rates
2. Compute Relative Difference Scores
3. Apply Bayesian smoothing

The result: a **smoothed score** for each template.

In [ ]:
policy = RDSPolicy(
    kappa=1000,   # Bayesian shrinkage strength
    gamma=0.1,    # Recency penalty magnitude
    h=0.5,        # Recency decay rate
    tau=50,       # Softmax temperature
)

policy.fit(train_df)

In [ ]:
# Policy summary
policy.summary()

# Bar chart of learned scores
templates = sorted(policy.smoothed_scores.keys())
scores = [policy.smoothed_scores[t] for t in templates]
colors = ["#2ecc71" if s > 0 else "#e74c3c" for s in scores]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(templates, scores, color=colors, edgecolor="white")
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_title("Learned Template Scores (Smoothed RDS)", fontsize=14, fontweight="bold")
ax.set_xlabel("Template")
ax.set_ylabel("Smoothed Score")
plt.tight_layout()
plt.show()

---
## 3. Offline Evaluation — The Concept

We can't deploy our policy live. Instead, we use **importance sampling** to estimate its performance from the test data (collected under random selection).

For each test event:

| Symbol | Meaning |
|--------|---------|
| $\pi_{\log}(a)$ | Probability the **logging policy** (random) selected template $a$ = $1 / |\text{eligible}|$ |
| $\pi_{\text{RDS}}(a)$ | Probability our **RDS policy** would select template $a$ |
| $w$ | Importance weight = $\pi_{\text{RDS}}(a) / \pi_{\log}(a)$ |

The **Weighted Importance Sampling** estimate:

$$\hat{V}(\pi_{\text{RDS}}) = \frac{\sum_i w_i \times r_i}{\sum_i w_i}$$

If $w > 1$: our policy is MORE likely to choose this template than random → upweight  
If $w < 1$: our policy is LESS likely → downweight

---
## 4. Evaluate the Policy

In [ ]:
results = policy.evaluate(test_df, max_weight=20, sample_size=10_000)

In [ ]:
# Results summary
print("\n" + "=" * 50)
print("         EVALUATION RESULTS")
print("=" * 50)
print(f"  Random Baseline:  {results['baseline']:.6f} ({results['baseline']:.2%})")
print(f"  RDS Policy:       {results['target_value']:.6f} ({results['target_value']:.2%})")
print(f"  Absolute Gain:    {results['target_value'] - results['baseline']:+.6f}")
print(f"  Relative Lift:    {results['lift']:+.2%}")
print(f"  Events Evaluated: {results['n_events']:,}")
print("=" * 50)

if results['lift'] > 0:
    print(f"\n✅ RDS OUTPERFORMS random by {results['lift']:.2%}")
else:
    print(f"\n⚠️ RDS underperforms random on this sample — try larger data or tune hyperparameters")

print(f"\n(Duolingo reported ~1.9% lift in their paper using the full dataset)")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(6, 5))

labels = ["Random\nBaseline", "RDS\nPolicy"]
values = [results["baseline"], results["target_value"]]
colors = ["#95a5a6", "#2ecc71" if results["lift"] > 0 else "#e74c3c"]

bars = ax.bar(labels, values, color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{val:.4f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_title(f"Random vs RDS (Lift: {results['lift']:+.2%})", fontsize=14, fontweight="bold")
ax.set_ylabel("Estimated Reward Rate")
plt.tight_layout()
plt.show()

---
## 5. Importance Weight Distribution

Let's look at the importance weights to understand how much our policy differs from random.

In [ ]:
# Re-compute weights for the evaluation sample so we can inspect them
from tqdm import tqdm
import ast

eval_df = test_df.sample(n=10_000, random_state=42).reset_index(drop=True)

target_probs_list = []
for i in range(len(eval_df)):
    eligible = eval_df["eligible_templates"].iloc[i]
    history = eval_df["history"].iloc[i]
    
    if isinstance(eligible, str):
        eligible = ast.literal_eval(eligible)
    elif hasattr(eligible, 'tolist'):
        eligible = eligible.tolist()
    
    if history is None:
        history = []
    elif isinstance(history, str):
        history = ast.literal_eval(history)
    elif hasattr(history, 'tolist'):
        history = history.tolist()
    
    probs = policy.get_probabilities(eligible, history)
    target_probs_list.append(probs)

weights = compute_importance_weights(eval_df, target_probs_list, max_weight=20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram
axes[0].hist(weights, bins=50, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].axvline(x=1.0, color="red", linestyle="--", label="w=1 (same as random)")
axes[0].axvline(x=weights.mean(), color="orange", linestyle="--", label=f"Mean={weights.mean():.2f}")
axes[0].set_title("Importance Weight Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Weight")
axes[0].set_ylabel("Count")
axes[0].legend()

# Right: weight vs reward
rewards = eval_df["session_end_completed"].values
axes[1].scatter(weights, rewards, alpha=0.1, s=5, color="#2c3e50")
axes[1].set_title("Weight vs Reward", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Importance Weight")
axes[1].set_ylabel("Reward (0 or 1)")

plt.tight_layout()
plt.show()

ess = (np.sum(weights)**2) / np.sum(weights**2)
print(f"Effective Sample Size: {ess:,.0f} / {len(weights):,} ({ess/len(weights):.1%})")
print(f"Higher ESS = more reliable estimate. Lower ESS = a few events dominate.")

---
## 6. Hyperparameter Sensitivity

How much does the lift change when we tweak the key hyperparameters?

In [ ]:
# Test a few combinations
configs = [
    {"kappa": 500,  "gamma": 0.1, "h": 0.5, "tau": 50},
    {"kappa": 1000, "gamma": 0.1, "h": 0.5, "tau": 50},   # default
    {"kappa": 5000, "gamma": 0.1, "h": 0.5, "tau": 50},
    {"kappa": 1000, "gamma": 0.1, "h": 0.5, "tau": 10},
    {"kappa": 1000, "gamma": 0.1, "h": 0.5, "tau": 100},
    {"kappa": 1000, "gamma": 0.05, "h": 0.5, "tau": 50},
    {"kappa": 1000, "gamma": 0.2, "h": 0.5, "tau": 50},
]

print(f"Testing {len(configs)} hyperparameter configurations...\n")
print(f"{'κ':>6} {'γ':>6} {'h':>6} {'τ':>6}  {'Baseline':>10} {'RDS Value':>10} {'Lift':>8}")
print("-" * 60)

hp_results = []
for cfg in configs:
    p = RDSPolicy(**cfg)
    p.fit(train_df)
    r = p.evaluate(test_df, max_weight=20, sample_size=10_000)
    
    print(f"{cfg['kappa']:>6} {cfg['gamma']:>6} {cfg['h']:>6} {cfg['tau']:>6}  "
          f"{r['baseline']:>10.6f} {r['target_value']:>10.6f} {r['lift']:>+8.2%}")
    hp_results.append({**cfg, **r})

In [ ]:
# Lift comparison chart
fig, ax = plt.subplots(figsize=(12, 5))

labels = [f"κ={c['kappa']}\nγ={c['gamma']}\nτ={c['tau']}" for c in configs]
lifts = [r["lift"] for r in hp_results]
colors = ["#2ecc71" if l > 0 else "#e74c3c" for l in lifts]

bars = ax.bar(range(len(labels)), [l * 100 for l in lifts], color=colors, edgecolor="white")
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_title("Lift (%) Across Hyperparameter Configurations", fontsize=14, fontweight="bold")
ax.set_ylabel("Lift (%)")

for bar, l in zip(bars, lifts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{l:+.2%}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

---
## 7. Final Summary — Everything We Built

| Notebook | Layer | What it does |
|----------|-------|-------------|
| **01** | Data | Explored dataset structure: 12 templates, sleeping arms, history, binary reward |
| **02** | Baseline | Random policy reward rate = the number to beat |
| **03** | RDS | Relative Difference Scores — removes confounding from user quality |
| **04** | Smoothing + Recency | Bayesian shrinkage (κ) + exponential decay penalty (γ, h) |
| **05** | Softmax | Temperature-controlled exploration/exploitation (τ) |
| **06** | Evaluation | Importance sampling to estimate policy value offline → lift |

**The full pipeline in one sentence:** Compute how much better each template performs compared to its peers (RDS), smooth noisy estimates (Bayesian), penalize recently-sent templates (recency), sample proportionally to scores (softmax), and measure improvement over random using importance weights.

### Next steps
1. Run on larger samples (500K, 1M, or full dataset) for more reliable estimates
2. Use the actual train/test split (separate parquet files) instead of sampling
3. Grid search over hyperparameters to find optimal (κ, γ, h, τ)
4. Compare against other baseline policies (e.g., greedy, epsilon-greedy)